<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/GES3_01_ClinVar_XML_Normalizer_PRODUCTION_BATCH2_2021Q2_COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/GES3_01_ClinVar_XML_Normalizer_PRODUCTION_BATCH2_2021Q2_COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# GES 3.0 — Notebook 01 Production Batch 2
## ClinVar XML Normalizer — Full Releases, 2021-04 → 2021-06

This notebook continues the validated **GES 3.0 Stage 01 production normalizer** into the second frozen longitudinal batch:

**April 2021 → June 2021**

It deliberately reuses the exact executable source of the already-validated Batch 1 notebook and applies only deterministic Batch-2 substitutions. The Stage-00 frozen manifest contract remains unchanged:

- frozen window: **2021-01 → 2026-08**
- 68 paired months
- 136 canonical model-month rows
- RCV: 37 legacy + 31 current
- VCV: 37 legacy + 31 current

### Why this wrapper exists

The goal is to prevent parser drift between production batches. Before any ClinVar source is parsed, the notebook:

1. fetches the validated Batch-1 notebook from the project repository;
2. records its SHA-256;
3. clears old outputs;
4. changes only Batch-1-specific labels, artifact names, decision keys, and the production interval;
5. verifies the Stage-00 freeze was not altered;
6. writes an expanded, audit-ready Batch-2 notebook;
7. executes the patched production cells in the current Colab kernel.


## 1. Fetch the validated Batch-1 source and construct Batch 2

This cell does **not** generically replace every occurrence of `2021-01` or `2021-03`. That would be unsafe because the Stage-00 freeze legitimately begins in 2021-01.

Only explicit Batch-1 configuration and provenance identifiers are changed.


In [1]:
from __future__ import annotations

import copy
import hashlib
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import requests

SOURCE_URL = (
    "https://raw.githubusercontent.com/SANGHATI23/"
    "genomic-evidence-reliability/main/"
    "GES3_01_ClinVar_XML_Normalizer_PRODUCTION_BATCH1_2021Q1_COLAB.ipynb"
)

TARGET_FILENAME = (
    "GES3_01_ClinVar_XML_Normalizer_PRODUCTION_BATCH2_2021Q2_EXPANDED.ipynb"
)
TARGET_PATH = Path("/content") / TARGET_FILENAME

print("Fetching validated Batch-1 source...")
resp = requests.get(SOURCE_URL, timeout=120)
resp.raise_for_status()
raw_bytes = resp.content

SOURCE_SHA256 = hashlib.sha256(raw_bytes).hexdigest()
print("Batch-1 source bytes:", len(raw_bytes))
print("Batch-1 source SHA-256:", SOURCE_SHA256)

if len(raw_bytes) < 250_000:
    raise RuntimeError(
        "Fetched Batch-1 notebook is unexpectedly small; refusing to continue."
    )

source_nb = json.loads(raw_bytes.decode("utf-8"))
patched_nb = copy.deepcopy(source_nb)

TARGET_GITHUB_NAME = (
    "GES3_01_ClinVar_XML_Normalizer_PRODUCTION_BATCH2_2021Q2_COLAB.ipynb"
)

patch_counts = {}

def replace_counted(text: str, old: str, new: str) -> str:
    n = text.count(old)
    if n:
        patch_counts[old] = patch_counts.get(old, 0) + n
        text = text.replace(old, new)
    return text

def patch_cell_source(src: str) -> str:
    s = src

    # Final "next batch" instruction must advance to Q3 before the accepted
    # Batch-2 interval is inserted elsewhere.
    s = replace_counted(
        s,
        "The next production batch should be:\n\n**2021-04 → 2021-06**",
        "The next production batch should be:\n\n**2021-07 → 2021-09**",
    )

    # Notebook / artifact provenance identifiers.
    replacements = [
        (
            "GES3_01_ClinVar_XML_Normalizer_PRODUCTION_BATCH1_2021Q1_COLAB.ipynb",
            TARGET_GITHUB_NAME,
        ),
        (
            "GES3_01_ClinVar_XML_Normalizer_PRODUCTION_BATCH1_2021Q1",
            "GES3_01_ClinVar_XML_Normalizer_PRODUCTION_BATCH2_2021Q2",
        ),
        ("GES3_STAGE01_BATCH1_2021Q1", "GES3_STAGE01_BATCH2_2021Q2"),
        ("stage01_batch1_2021Q1", "stage01_batch2_2021Q2"),
        ("GES3_01_production_batch1", "GES3_01_production_batch2"),
        ("batch1_go", "batch2_go"),
        ("PRODUCTION BATCH 1", "PRODUCTION BATCH 2"),
        ("Production Batch 1", "Production Batch 2"),
        ("Batch 1", "Batch 2"),
        ("first frozen longitudinal batch", "second frozen longitudinal batch"),
        ("January 2021 → March 2021", "April 2021 → June 2021"),
        ("2021-01 → 2021-03", "2021-04 → 2021-06"),
    ]
    for old, new in replacements:
        s = replace_counted(s, old, new)

    # Executable interval: replace ONLY the Batch variables, not Stage-00.
    s = replace_counted(
        s,
        'BATCH_START_MONTH = "2021-01"',
        'BATCH_START_MONTH = "2021-04"',
    )
    s = replace_counted(
        s,
        'BATCH_END_MONTH = "2021-03"',
        'BATCH_END_MONTH = "2021-06"',
    )

    return s

for cell in patched_nb.get("cells", []):
    src = cell.get("source", "")
    if isinstance(src, list):
        src = "".join(src)

    cell["source"] = patch_cell_source(src)

    # Never carry Batch-1 execution evidence into Batch 2.
    if cell.get("cell_type") == "code":
        cell["execution_count"] = None
        cell["outputs"] = []

patched_nb.setdefault("metadata", {})
patched_nb["metadata"]["ges3_batch_patch"] = {
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "source_url": SOURCE_URL,
    "source_sha256": SOURCE_SHA256,
    "source_batch": "2021-01_to_2021-03",
    "target_batch": "2021-04_to_2021-06",
    "stage00_freeze": "2021-01_to_2026-08",
    "method": "deterministic_batch_specific_source_substitution",
}

print("\nPatch targets changed:")
for k, v in sorted(patch_counts.items()):
    print(f"  {v:>3} × {k!r}")


Fetching validated Batch-1 source...
Batch-1 source bytes: 374734
Batch-1 source SHA-256: c9ee82f07e72f65291cbc73df058360afc23593ceddab8c15f0f88173e4766cd

Patch targets changed:
    3 × '2021-01 → 2021-03'
    1 × 'BATCH_END_MONTH = "2021-03"'
    1 × 'BATCH_START_MONTH = "2021-01"'
    6 × 'Batch 1'
    1 × 'GES3_01_ClinVar_XML_Normalizer_PRODUCTION_BATCH1_2021Q1_COLAB.ipynb'
    1 × 'GES3_01_production_batch1'
    2 × 'GES3_STAGE01_BATCH1_2021Q1'
    1 × 'January 2021 → March 2021'
    1 × 'PRODUCTION BATCH 1'
    4 × 'Production Batch 1'
    1 × 'The next production batch should be:\n\n**2021-04 → 2021-06**'
    2 × 'batch1_go'
    1 × 'first frozen longitudinal batch'
   12 × 'stage01_batch1_2021Q1'


## 2. Hard preflight validation

These assertions are intentionally strict. If any one fails, **do not run the production parser**.

The checks confirm that:

- Batch 2 is exactly **2021-04 → 2021-06**;
- the Stage-00 lock remains **2021-01 → 2026-08**;
- the expected three-month / six-source batch structure remains intact;
- Batch-1 artifact and decision identifiers do not remain in executable code.


In [2]:
all_source = "\n".join(
    "".join(c.get("source", []))
    if isinstance(c.get("source", []), list)
    else str(c.get("source", ""))
    for c in patched_nb["cells"]
)

code_source = "\n".join(
    "".join(c.get("source", []))
    if isinstance(c.get("source", []), list)
    else str(c.get("source", ""))
    for c in patched_nb["cells"]
    if c.get("cell_type") == "code"
)

# Target batch must be exact.
assert code_source.count('BATCH_START_MONTH = "2021-04"') == 1
assert code_source.count('BATCH_END_MONTH = "2021-06"') == 1
assert 'BATCH_START_MONTH = "2021-01"' not in code_source
assert 'BATCH_END_MONTH = "2021-03"' not in code_source

# Stage-00 frozen audit invariants MUST remain untouched.
assert 'FROZEN_START_MONTH = "2021-01"' in code_source
assert 'FROZEN_END_MONTH = "2026-08"' in code_source
assert "EXPECTED_FROZEN_MONTHS = 68" in code_source
assert "EXPECTED_CANONICAL_ROWS = 136" in code_source
assert '("RCV", "legacy"): 37' in code_source
assert '("RCV", "current"): 31' in code_source
assert '("VCV", "legacy"): 37' in code_source
assert '("VCV", "current"): 31' in code_source

# Three months × two products = six production sources.
assert "EXPECTED_BATCH_MONTHS = 3" in code_source
assert "EXPECTED_BATCH_SOURCES = EXPECTED_BATCH_MONTHS * len(MODELS_TO_PROCESS)" in code_source
assert 'MODELS_TO_PROCESS = ["RCV", "VCV"]' in code_source

# Batch-2 provenance/decision names must exist.
assert "stage01_batch2_2021Q2" in code_source
assert "GES3_01_production_batch2" in code_source
assert "batch2_go" in code_source

# No executable Batch-1 provenance may survive.
for forbidden in [
    "stage01_batch1_2021Q1",
    "GES3_01_production_batch1",
    "batch1_go",
]:
    assert forbidden not in code_source, f"Unsafe leftover identifier: {forbidden}"

# Preserve production behavior.
assert 'PILOT_RECORD_LIMIT_PER_SOURCE = None' in code_source
assert "RESUME_COMPLETED_RELEASES = True" in code_source
assert "DELETE_INCOMPLETE_PARTITION_BEFORE_RETRY = True" in code_source
assert "HASH_PARQUET_SHARDS = True" in code_source

TARGET_PATH.write_text(
    json.dumps(patched_nb, ensure_ascii=False, indent=1),
    encoding="utf-8",
)

target_sha = hashlib.sha256(TARGET_PATH.read_bytes()).hexdigest()

print("BATCH-2 PREFLIGHT: PASS")
print("Target interval: 2021-04 → 2021-06")
print("Stage-00 lock: 2021-01 → 2026-08")
print("Expanded audit notebook:", TARGET_PATH)
print("Expanded notebook SHA-256:", target_sha)


BATCH-2 PREFLIGHT: PASS
Target interval: 2021-04 → 2021-06
Stage-00 lock: 2021-01 → 2026-08
Expanded audit notebook: /content/GES3_01_ClinVar_XML_Normalizer_PRODUCTION_BATCH2_2021Q2_EXPANDED.ipynb
Expanded notebook SHA-256: 9259f6f4eb2d18a7bd554048f463de5e5d80c09b3bf7cd3ee75411c0ccaaa064


## 3. Execute the validated Batch-2 production normalizer

Running this cell executes the patched source cells **in their original order** in the current Colab kernel.

Expected production behavior is therefore the same as validated Batch 1:

- Stage-00 manifest lock / strict reconstruction check
- six selected sources: RCV + VCV for April, May, June 2021
- full-stream XML parsing
- compressed-stream MD5/SHA-256
- NCBI MD5 verification
- Parquet sharding
- completion markers
- structural QC
- legacy classification-semantics audit
- RCV↔SCV relationship audit
- artifact hashing
- final `batch2_go` decision

Google Drive persistence remains enabled exactly as in Batch 1.


In [3]:
from IPython import get_ipython

shell = get_ipython()
if shell is None:
    raise RuntimeError("This production runner requires an IPython/Colab kernel.")

source_code_cells = [
    (i, c)
    for i, c in enumerate(patched_nb["cells"])
    if c.get("cell_type") == "code"
]

print("Executable patched source cells:", len(source_code_cells))
print("Starting GES 3.0 Stage 01 — Production Batch 2")
print("Interval: 2021-04 → 2021-06")
print("=" * 72)

for ordinal, (source_index, cell) in enumerate(source_code_cells, start=1):
    src = cell.get("source", "")
    if isinstance(src, list):
        src = "".join(src)

    first_nonempty = next(
        (line.strip() for line in src.splitlines() if line.strip()),
        "<empty cell>",
    )

    print(
        f"\n\n######## VALIDATED SOURCE CELL "
        f"{ordinal}/{len(source_code_cells)} "
        f"(original index {source_index}) ########"
    )
    print(first_nonempty[:140])

    result = shell.run_cell(src, store_history=False)

    err = getattr(result, "error_before_exec", None) or getattr(
        result, "error_in_exec", None
    )
    if err is not None:
        raise RuntimeError(
            f"Batch-2 execution failed in validated source cell "
            f"{ordinal} (original notebook index {source_index})."
        ) from err

print("\n" + "=" * 72)
print("All patched Batch-2 source cells executed.")


Executable patched source cells: 25
Starting GES 3.0 Stage 01 — Production Batch 2
Interval: 2021-04 → 2021-06


######## VALIDATED SOURCE CELL 1/25 (original index 3) ########
!pip -q install lxml pyarrow pandas requests tqdm beautifulsoup4


######## VALIDATED SOURCE CELL 2/25 (original index 5) ########
from __future__ import annotations
GES 3.0 Stage 01 started: 2026-08-16T19:48:34.297127+00:00
Python: 3.12.13
lxml: 6.1.1
pyarrow: 18.1.0
pandas: 2.2.2


######## VALIDATED SOURCE CELL 3/25 (original index 7) ########
# ============================================================
RUN_PROFILE = PRODUCTION_BATCH
Batch = 2021-04 → 2021-06
Models = ['RCV', 'VCV']
Google Drive = True


######## VALIDATED SOURCE CELL 4/25 (original index 9) ########
if USE_GOOGLE_DRIVE:
Mounted at /content/drive
ROOT: /content/drive/MyDrive/GES3
Stage 00 directory: /content/drive/MyDrive/GES3/stage00
Stage 01 output: /content/drive/MyDrive/GES3/stage01


######## VALIDATED SOURCE CELL 5/25 (original index 

,data_model,format_generation,n
0,RCV,current,31
1,RCV,legacy,37
2,VCV,current,31
3,VCV,legacy,37




######## VALIDATED SOURCE CELL 7/25 (original index 15) ########
def month_between(s: pd.Series, start: str, end: str) -> pd.Series:
Batch source files: 6
Record limit: None
Batch manifest SHA-256: e729032ea47eced4b090a387a9c40881a54a983e23d5e1f47a413886e5c70e6a


,release_month,data_model,format_generation,url,ncbi_md5
0,2021-04,RCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,8521631ded8ef0da2dc8ec318a656771
1,2021-04,VCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,6158516dc0f79c023a423e52ed95ff14
2,2021-05,RCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,cb429b4e4dcf932466501781071f5a6e
3,2021-05,VCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,bbf2716f8d2e44736ed6e9e3a5be6160
4,2021-06,RCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,d4883d850b8e710982e6347cdaa26231
5,2021-06,VCV,legacy,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,1e4b6ab71d35b77905d13cf052ccff90




######## VALIDATED SOURCE CELL 8/25 (original index 17) ########
GERMLINE_TERMS = {
Normalized schema contract loaded.
vcv_state columns: 25
rcv_state columns: 25
scv_state columns: 23
vcv_rcv_link columns: 7


######## VALIDATED SOURCE CELL 9/25 (original index 19) ########
def localname(tag) -> str:
XML utilities loaded.


######## VALIDATED SOURCE CELL 10/25 (original index 21) ########
def parse_vcv_record(elem, release_month, source_url, source_format):


######## VALIDATED SOURCE CELL 11/25 (original index 23) ########
def infer_submission_meta(assertion):
SCV parser loaded.


######## VALIDATED SOURCE CELL 12/25 (original index 25) ########
def parse_rcv_record(elem, release_month, source_url, source_format):


######## VALIDATED SOURCE CELL 13/25 (original index 27) ########
class ShardWriter:
Parquet shard writer loaded.


######## VALIDATED SOURCE CELL 14/25 (original index 29) ########
class HashingReader:
Production streaming hash wrapper loaded.


######## VALIDATED SOUR

...


 2021-06 RCV legacy rcv_state


,release_month,source_url,source_format,rcv_accession,rcv_version,vcv_accession,vcv_version,variation_id,variation_name,germline_description,...,oncogenicity_review_status,legacy_classification_description,legacy_review_status,legacy_date_last_evaluated,legacy_germline_candidate,condition_names_json,condition_ids_json,gene_symbols_json,citation_ids_json,scv_count
0,2021-06,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,legacy,RCV000000470,4,VCV000000441,1,441,None,None,...,None,Pathogenic,no assertion criteria provided,2007-03-01,True,"[""BLOOD GROUP--LUTHERAN NULL""]","[""OMIM:612773.0003"", ""OMIM:612773.0004"", ""OMIM...",[],"[""PubMed:17319831""]",1
1,2021-06,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,legacy,RCV000003355,3,VCV000003202,1,3202,None,None,...,None,Pathogenic,no assertion criteria provided,1999-11-01,True,"[""Congenital muscular dystrophy-dystroglycanop...","[""OMIM:607440.0001"", ""OMIM:607440.0002"", ""OMIM...",[],"[""PubMed:10545611""]",1
2,2021-06,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,legacy,RCV000003367,3,VCV000003212,1,3212,None,None,...,None,Pathogenic,no assertion criteria provided,2008-02-01,True,"[""Congenital muscular dystrophy-dystroglycanop...","[""OMIM:607440.0001"", ""OMIM:607440.0002"", ""OMIM...",[],"[""PubMed:18177472""]",1
3,2021-06,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,legacy,RCV000004080,2,VCV000003876,1,3876,None,None,...,None,Pathogenic,no assertion criteria provided,1990-10-15,True,"[""Hexosaminidase B (paris)""]","[""MedGen:C4016989""]",[],"[""PubMed:2170400"", ""PubMed:868875""]",1
4,2021-06,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/R...,legacy,RCV000005054,3,VCV000004788,1,4788,None,None,...,None,Pathogenic,no assertion criteria provided,2001-02-01,True,"[""Autosomal recessive Dejerine-Sottas syndrome""]","[""MedGen:CN069172""]",[],"[""PubMed:11133365""]",1



 2021-06 VCV legacy vcv_state


,release_month,source_url,source_format,vcv_accession,vcv_version,variation_id,variation_type,variation_name,date_created,date_last_updated,...,oncogenicity_description,oncogenicity_review_status,legacy_classification_description,legacy_review_status,legacy_date_last_evaluated,legacy_germline_candidate,gene_symbols_json,rcv_accessions_json,citation_ids_json,scv_count
0,2021-06,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,legacy,VCV000000441,1,441,Deletion,"BCAM, EX3-4DEL",2010-12-01,2019-03-29,...,None,None,Pathogenic,None,2007-03-01,True,"[""BCAM""]","[""RCV000000470""]","[""PubMed:17319831""]",1
1,2021-06,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,legacy,VCV000003202,1,3202,Insertion,"FKTN, L1 INS",2010-12-01,2019-03-29,...,None,None,Pathogenic,None,1999-11-01,True,"[""FKTN""]","[""RCV000003355""]","[""PubMed:10545611""]",1
2,2021-06,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,legacy,VCV000003212,1,3212,Deletion,"FKTN, 473-BP DEL, NT5370",2010-12-01,2019-03-29,...,None,None,Pathogenic,None,2008-02-01,True,"[""FKTN""]","[""RCV000003367""]","[""PubMed:18177472""]",1
3,2021-06,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,legacy,VCV000003876,1,3876,Insertion,"HEXB, 18-BP INS",2010-12-01,2019-03-29,...,None,None,Pathogenic,None,1990-10-15,True,"[""HEXB""]","[""RCV000004080""]","[""PubMed:2170400"", ""PubMed:868875""]",1
4,2021-06,https://ftp.ncbi.nlm.nih.gov/pub/clinvar/xml/V...,legacy,VCV000004788,1,4788,Deletion,"PRX, 1-BP DEL, 2787C",2010-12-01,2019-03-29,...,None,None,Pathogenic,None,2001-02-01,True,"[""PRX""]","[""RCV000005054""]","[""PubMed:11133365""]",1



All patched Batch-2 source cells executed.


## 4. Final Batch-2 acceptance check and source provenance copy

A run is accepted only if the normalizer's own final decision reports:

**`batch2_go = true`**

The expanded source used for the run is also copied into Stage-01 metadata after Drive has been mounted, so the exact executable code is retained with the production artifacts.


In [4]:
if "decision" not in globals():
    raise RuntimeError("Production decision object was not created.")

print(json.dumps(decision, indent=2))

if decision.get("batch2_go") is not True:
    raise RuntimeError(
        "Batch 2 is NOT accepted. Inspect source failures and QC before continuing."
    )

# Persist the exact expanded executable source beside Stage-01 metadata.
if "META_DIR" in globals():
    provenance_path = Path(META_DIR) / (
        "GES3_01_ClinVar_XML_Normalizer_"
        "PRODUCTION_BATCH2_2021Q2_EXPANDED_SOURCE.ipynb"
    )
    provenance_path.write_bytes(TARGET_PATH.read_bytes())

    provenance_sha = hashlib.sha256(
        provenance_path.read_bytes()
    ).hexdigest()

    sha_path = provenance_path.with_suffix(
        provenance_path.suffix + ".sha256"
    )
    sha_path.write_text(
        provenance_sha + "  " + provenance_path.name + "\n",
        encoding="utf-8",
    )

    print("Persisted executable source:", provenance_path)
    print("Executable source SHA-256:", provenance_sha)

print("\nBATCH 2 ACCEPTED ✅")
print("2021-04 → 2021-06 complete.")
print("Next Stage-01 production interval: 2021-07 → 2021-09")


{
  "stage": "GES3_01_production_batch2",
  "batch": "2021-04_to_2021-06",
  "expected_sources": 6,
  "verified_completed_sources": 6,
  "failed_sources": 0,
  "all_six_completed": true,
  "structural_qc_passed": true,
  "legacy_semantics_ok": true,
  "rcv_scv_relationship_ok": true,
  "batch2_go": true
}
Persisted executable source: /content/drive/MyDrive/GES3/stage01/metadata/GES3_01_ClinVar_XML_Normalizer_PRODUCTION_BATCH2_2021Q2_EXPANDED_SOURCE.ipynb
Executable source SHA-256: 9259f6f4eb2d18a7bd554048f463de5e5d80c09b3bf7cd3ee75411c0ccaaa064

BATCH 2 ACCEPTED ✅
2021-04 → 2021-06 complete.
Next Stage-01 production interval: 2021-07 → 2021-09
